# Handling Dirty Data

So far, the datasets we have dealt with have been mostly "clean". Clean data is the exception, not the rule. Most, if not all, of the time you have to deal with "dirty data". Specific issues that you encounter with dirty data include, but are not limited to:

- Missing data
- Duplicated data
- Inconsistent data within a column

This topic is very broad and we are only going to scratch the surface of it. You could spend an entire course just on handling dirty data. Becoming efficient at the process takes lots of practice and experience you can gain only by doing it. There are scenarios where one approach is much better suited than another. You must always keep in mind the end goal and the repercussions of the choices you make. You should explicitly document which steps you took, the order of those steps, and why you chose particular actions.

In this section we are going to work through an example of dealing with a dirty dataset. The dataset is somewhat "large", with more than 28,000 rows of data. A dataset with this many rows makes it very difficult to examine the data manually, so we will have to rely on our Python and `pandas` skills to find any issues and resolve them.

## Finding the Problematic Data

Our dataset is in the file named `dirty_data.csv` and contains hourly weather data for Williamsburg, VA. Our goal is to clean the data to make it easier to analyze and understand. Let's start exploring it.

In [ ]:
import pandas as pd

In [ ]:
# Read file into DataFrame named dirty_data
dirty_data = pd.read_csv('./data/dirty_data.csv')

# Look at the first few rows
dirty_data.head()

In [ ]:
# Get the summary statistics for numerical columns
dirty_data.describe()

Interesting. Only six out of the 18 columns appear to be numerical. I would have expected at least the other temperature columns to all be numerical. This means we had a lot of columns with the data type `object`. We can use `.describe(include='object')` to examine those columns more closely.

In [ ]:
# Use .describe(include='object') to examine those columns
dirty_data.describe(include='object')

If you want to examine any of these columns further, another step you can take is to see how frequently each value shows up for a particular column. You can accomplish this task by using `.value_counts()` on the desired column. Let's look at two columns: `HOURLYWindDirection` which has 38 unique values and `HOURLYDewPointTempF` which has 54 unique values.

In [ ]:
# How many of each value for HOURLYWindDirection?
dirty_data.HOURLYWindDirection.value_counts()

In [ ]:
# How many of each value for HOURLYDewPointTempF?
dirty_data.HOURLYDewPointTempF.value_counts()

### Missing Data

Do we have any missing data for any particular columns? Perhaps you noticed in our original `.describe()` call above that the column `HOURLYPRSENTWEATHERTYPE` had a count of 0. What does that mean? That entire column is empty or blank. Assuming the column was not important for our analysis, this is an easy fix. We can simply drop the column. If the column is important, then we need to determine why the column is blank and how to get the data.

We have several options to help us find any other missing data. We can use our good friend `.info()` which tells us how many "non-null" rows there are for each column. If this number is different than the total number of rows, then we have missing data for that column. There are also the `.isna()` and `.isnull()` functions that can be used on both `Series` and `DataFrame` objects. The functions do exactly the same thing. In fact, `.isnull` is an alias for `.isna`.

In [ ]:
# See the .info()
dirty_data.info()

In [ ]:
# How many missing values for each COLUMN
dirty_data.isna().sum()

In [ ]:
# How many rows have at least one missing value in any column?
dirty_data.isna().any(axis=1).sum()

So, every row has at least one missing value. Actually, this should not surprise you. We know the that column `HOURLYPRSENTWEATHERTYPE` was completely blank. We also see that the column `HOURLYWindGustSpeed` has about 87% of the rows missing data. Let's drop both of those columns and then try to find how many rows have at least one column with missing data.

In [ ]:
# Drop the columns `HOURLYPRSENTWEATHERTYPE` and `HOURLYWindGustSpeed`
dirty_data.drop(columns=['HOURLYPRSENTWEATHERTYPE', 'HOURLYWindGustSpeed'], inplace=True)

# See if it worked
dirty_data.info()

In [ ]:
# How many missing values for each COLUMN
dirty_data.isna().sum()

In [ ]:
# How many rows have at least one missing value in any column?
dirty_data.isna().any(axis=1).sum()

In [ ]:
# Find those rows
missing_rows = dirty_data[dirty_data.isna().any(axis=1)]
print(f'There were {missing_rows.shape[0]} rows that contain missing data')
missing_rows

### Duplicate Data 

Sometimes a dataset will contain rows of data that are duplicated. That is, the values in one row for every column are exactly the same for another row. The function `.duplicated()` helps us find the rows that are repeated. Let's try it.

In [ ]:
# What does .duplicated() return?
dirty_data.duplicated()

In [ ]:
# Get the rows that are True see how many there are
dirty_data[dirty_data.duplicated()].shape

In [ ]:
# Store the duplicates in the variable `dupes`
dupes = dirty_data[dirty_data.duplicated()]
dupes

By default the `.duplicated()` function marks duplicates as `True`, except for the first occurrence. So, what we are seeing above is the second occurrence of each row. What if we want to get back every row that is duplicated, including the initial row? We can use the argument `keep=False` within the `.duplicated()` function. Let's try it.

In [ ]:
dirty_data[dirty_data.duplicated(keep=False)]

In this case, it appears that the rows that are duplicated are next to each other. For example, index 59 and 60 are duplicates and we know they are next to each other based on their indices. 

<hr style="border:1px solid gray">

## Fixing the Issues

We now want to try to fix the issues we found. We demonstrate only one way to tackle these issues. Depending on the end goal, the amount of data you have, etc., other approaches may be more applicable. 

Our end goal is to create a dataset that contains both the Fahrenheit and Celsius temperatures. Here are the steps we are going to take.

1. We have already dropped two columns that were either completely empty or nearly empty.
2. Remove the duplicate rows. 
3. Drop any rows that contain null values in any of the temperature columns. 
4. Make sure that all the values in the temperature columns are numerical and those columns have a data type of `float64`.
5. Make sure the column `HOURLYStationPressure` is numerical.
6. Create new Celsius equivalent columns.
7. Change the value 'VRB' to 'Variable Direction' found in the column `HOURLYWindDirection`. 
8. Save the results to a file named `cleaned_data.csv`. 

### Step 2

We want to remove the duplicate rows. We know that there are six duplicates. This task is actually quite easy because there is a function called `.drop_duplicates()` that handles all the hard work for us. By default, this function only removes duplicate rows based on **all** columns. This is what we want, but if you want to remove duplicates on specific column(s), you can use the `subset` argument. You can also use the argument `keep` to determine which duplicate to keep. By default, it keeps the first occurrence and drops all the other duplicates. The defaults are all we need for our dataset. We need to capture the returned `DataFrame` into a new variable and work with it. Let's try it.

In [ ]:
# Print out the shape before trying to remove duplicates
dirty_data.shape

In [ ]:
# Drop the duplicate rows and save in `cleaned_data`
cleaned_data = dirty_data.drop_duplicates()

In [ ]:
# Shape of dirty_data -- should not have changed
dirty_data.shape

In [ ]:
# Shape of cleaned_data -- should be 6 less than before
cleaned_data.shape

### Step 3

We want to drop any rows that contain null values in any of the temperature columns. The function `.dropna()` removes missing values. Best practice indicates that you should always explicitly use the argument `axis` to specify whether you want to drop rows, `axis=0` or `axis='rows'`, or columns, `axis=1` or `axis='columns'` that contain missing values.

The three temperature columns are `HOURLYDRYBULBTEMPF`, `HOURLYWETBULBTEMPF`, and `HOURLYDewPointTempF`. Let's first look at how many possible rows will be removed. Then we will remove them and see the new shape of our dataset.

In [ ]:
# How many possible rows are we removing?
cleaned_data[['HOURLYDRYBULBTEMPF', 'HOURLYWETBULBTEMPF', 'HOURLYDewPointTempF']].isna().sum()

In [ ]:
# Drop the rows with missing values in the temperature columns
cleaned_data = cleaned_data.dropna(axis='rows',
                                   subset=['HOURLYDRYBULBTEMPF', 'HOURLYWETBULBTEMPF', 'HOURLYDewPointTempF'])

In [ ]:
# See how many rows are left (started with 28,531)
cleaned_data.shape

### Step 4

Now, we want to make sure that all the values in the temperature columns are numerical and those columns have a data type of `float64`. One of them, `HOURLYWETBULBTEMPF`, is already this way. The other two columns have a data type of `object`. Let's go ahead and examine all three columns a little closer. We will start by looking at all the unique values in each column. 

In [ ]:
# What are the unique values in HOURLYDRYBULBTEMPF
cleaned_data.HOURLYDRYBULBTEMPF.unique()

In [ ]:
# What are the unique values in HOURLYWETBULBTEMPF
cleaned_data.HOURLYWETBULBTEMPF.unique()

In [ ]:
# What are the unique values in HOURLYDewPointTempF
cleaned_data.HOURLYDewPointTempF.unique()

You should have noticed that the issue with both of the two troubled columns stems from the fact that there is the letter `s` at the end of several values. We would like to remove the `s` and keep the numerical portion. We do **not** want to hard-code specific values to search for. For example, we should not explicitly look for the value `52s`. Instead, we want a process that will find the letter `s` and remove it. An even more flexible approach would be to strip off *any* alphabetical characters instead of only focusing on the letter `s`. I am going to show you the flexible way to do it. We will have to use a *regular expression* within the `.replace()` function. Regular expressions are a huge and vitally important topic that you should investigate on your own.

In [ ]:
# Import the regular expression package, re
import re

In [ ]:
# Need to specify what we want to replace.
# I want to replace anything that is not a digit nor a period.
# We are replacing it with the empty string ''.
# We have to tell replace() that we want the matching to be treated as a regular expression.
# Finally, we are doing the replacement in place.
cleaned_data['HOURLYDRYBULBTEMPF'].replace(to_replace='[^0-9\.]', value='', regex=True, inplace=True)
cleaned_data['HOURLYDewPointTempF'].replace(to_replace='[^0-9\.]', value='', regex=True, inplace=True)

In [ ]:
# See the shape
cleaned_data.shape

In [ ]:
# Make sure we got rid of all the letters
cleaned_data.HOURLYDRYBULBTEMPF.unique()

In [ ]:
# Check other column too
cleaned_data.HOURLYDewPointTempF.unique()

In [ ]:
# See the .info()
cleaned_data.info()

Notice that even though we got rid of the letters in the values for those two columns, we still have strings and the columns are still of type `object`. We want to cast those two columns to decimals. 

In [ ]:
# Cast both columns to float64
cleaned_data.HOURLYDRYBULBTEMPF = cleaned_data.HOURLYDRYBULBTEMPF.astype('float64')
cleaned_data.HOURLYDewPointTempF = cleaned_data.HOURLYDewPointTempF.astype('float64')

In [ ]:
# Look at info() again
cleaned_data.info()

### Step 5

Now we want to convert the column `HOURLYStationPressure` to numerical. If we look at the unique values in this column, we will see that there are a few values that have the letter `s` at the end of the number. We could repeat the steps that we took above for the temperature columns. Instead, we are going to try to **coerce** this column to numerical by using the function `pd.to_numeric()`. Using the default arguments will raise errors. We have to explicitly tell the function that we want `errors='coerce'`. This will replace any values that cannot be converted to a number with null. Let's try it.

In [ ]:
# See the unique values of the column
cleaned_data.HOURLYStationPressure.unique()

In [ ]:
# Try default arguments for pd.to_numeric -- will raise errors
pd.to_numeric(cleaned_data.HOURLYStationPressure)

In [ ]:
# Now set errors to 'coerce'; put the numbers back into the original column
cleaned_data.HOURLYStationPressure = pd.to_numeric(cleaned_data.HOURLYStationPressure,
                                                   errors='coerce')

In [ ]:
# Look at info() to see if column data type changed
cleaned_data.info()

In [ ]:
# See if the null values are in unique values
cleaned_data.HOURLYStationPressure.unique()

### Step 6

Now we need to create new Celsius equivalent columns for the three temperature columns. We could accomplish this task by using the `.assign()` function. However, to show you another way to accomplish this task, we are going to create a function that does the conversion. We will call the function three times, once for each column.

In [ ]:
# Define a function called f2c
def f2c(f):
    return (f-32.0) * 5/9

In [ ]:
# Now use the function to create the new columns
cleaned_data['HOURLYDRYBULBTEMPC'] = f2c(cleaned_data['HOURLYDRYBULBTEMPF'])
cleaned_data['HOURLYWETBULBTEMPC'] = f2c(cleaned_data['HOURLYWETBULBTEMPF'])
cleaned_data['HOURLYDewPointTempC'] = f2c(cleaned_data['HOURLYDewPointTempF'])

In [ ]:
# Look at info()
cleaned_data.info()

In [ ]:
# Sample a few rows
cleaned_data.sample(10)

### Step 7

Now we want to change the value 'VRB' found in the column `HOURLYWindDirection` to 'Variable Direction'. Again, there are several ways to accomplish this task. Here, we will create another user-defined function and then use the `.apply()` function with our user-defined function.

In [ ]:
# Define a function to convert 'VRB' to 'Variable Direction'
# This function assumes you send it the entire row of a DataFrame
def vrb(row):
    if row['HOURLYWindDirection'] == 'VRB':
        row['HOURLYWindDirection'] = 'Variable Direction'
    return row

In [ ]:
# Here is a lambda function that expects you to send it a
# single column (Series), in particular the 'HOURLYWindDirection' column
vrb2 = lambda x: 'Variable Direction' if x == 'VRB' else x

In [ ]:
# make a copy to use to try out first function
cleaned_data2 = cleaned_data.copy()

In [ ]:
# Make sure HOURLYWindDirection has 'VRB'
cleaned_data['HOURLYWindDirection'].unique()

In [ ]:
# Make sure HOURLYWindDirection has 'VRB'
cleaned_data2['HOURLYWindDirection'].unique()

In [ ]:
# Try using the first function vrb, using the entire DataFrame
cleaned_data2 = cleaned_data2.apply(vrb, axis='columns')

In [ ]:
# See if it worked
cleaned_data2.HOURLYWindDirection.unique()

In [ ]:
# Use the second function, which expects a Series
cleaned_data['HOURLYWindDirection'] = cleaned_data['HOURLYWindDirection'].apply(vrb2)
cleaned_data.info()

In [ ]:
cleaned_data.HOURLYWindDirection.unique()

### Step 8 
Finally, let's save the results to a file named `cleaned_data.csv`. 

In [ ]:
# Save results to cleaned_data.csv
cleaned_data.to_csv('./data/cleaned_data.csv', index=False)

### Additional Resources

The following links point you to additional resources that you might find helpful in learning this material.

1. [The API reference for `pandas.DataFrame.duplicated`][1].
2. [The API reference for `pandas.DataFrame.drop_duplicates`][2]
3. [The API reference for `pandas.DataFrame.dropna`][3]
4. [The API reference for `pandas.DataFrame.apply`][4]
5. [The `pandas` User Guide][5].


-----

[1]: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html
[2]: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html
[3]: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html
[4]: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html
[5]: https://pandas.pydata.org/docs/user_guide/index.html

-----

**&copy; 2022 - Present: Matthew D. Dean, Ph.D.   
Clinical Associate Professor of Business Analytics at William \& Mary.**